<a href="https://colab.research.google.com/github/sangjkim930/AI-Driven-Research-Methodology/blob/main/03_Reusable_Research_Knowledge_Base.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Building a Reusable Research Knowledge Base

**5 Papers → Vector Store → Ask → Add 1 Paper → Ask Again → Evidence File**

Run the cells in order from top to bottom.

### Step 1. Set Up the Environment

Connect to the OpenAI API and choose the model used for the research tasks.

In [1]:
!pip install -q openai

from openai import OpenAI
from google.colab import userdata, files

client = OpenAI(
    api_key=userdata.get("OPENAI_API_KEY")
)

ANSWER_MODEL = "gpt-5.6-luna"

### Step 2. Create the Research Knowledge Base

Upload five papers, create a vector store, and add the papers to it.

In [ ]:
# import glob
# pdf_files = sorted(glob.glob("/content/*.pdf"))
# paper_files = pdf_files

uploaded = files.upload()
paper_files = list(uploaded.keys())

vector_store = client.vector_stores.create(
    name="Research Knowledge Base"
)

for file_name in paper_files:
    client.vector_stores.files.upload_and_poll(
        vector_store_id=vector_store.id,
        file=open(file_name, "rb")
    )

print("Papers added:", len(paper_files))
print("Vector Store ID:", vector_store.id)

### Step 3. Ask a Research Question

Ask a question using only the papers stored in the vector store.

In [ ]:
QUESTION = """
Compare the relationship between environmental performance
and financial performance across the papers.

Use only the papers in the research knowledge base.
Distinguish each paper's own empirical findings from
prior studies cited in the paper.
If the evidence is insufficient, say so.
"""

response = client.responses.create(
    model=ANSWER_MODEL,
    input=QUESTION,
    tools=[
        {
            "type": "file_search",
            "vector_store_ids": [vector_store.id]
        }
    ]
)

print(response.output_text)

### Step 4. Add One New Paper

Upload one additional paper and add it to the same vector store.

In [ ]:
new_upload = files.upload()
new_file = list(new_upload.keys())[0]

client.vector_stores.files.upload_and_poll(
    vector_store_id=vector_store.id,
    file=open(new_file, "rb")
)

print("New paper added:", new_file)

### Step 5. Ask the Same Question Again

Run the same question after adding the new paper and compare the result.

In [ ]:
updated_response = client.responses.create(
    model=ANSWER_MODEL,
    input=QUESTION,
    tools=[
        {
            "type": "file_search",
            "vector_store_ids": [vector_store.id]
        }
    ]
)

print(updated_response.output_text)

### Step 6. Automate Structured Research Extraction

Apply the same extraction rules across all papers, organize the results as structured data, and save them as a CSV file.

In [ ]:
import json
import pandas as pd

MATRIX_PROMPT = """
Using only the papers in the research knowledge base,
create a research evidence matrix.

Return JSON only as a list of records with these fields:

Paper
Country_Sample
Study_Period
Method
Environmental_Performance_Measure
Financial_Performance_Measure
Relationship
Main_Finding

Use the actual measures reported in each paper.
For Relationship, use:
Positive, Negative, Insignificant, Nonlinear, or Mixed.

Report each paper's own empirical findings, not prior studies
cited in the paper.

If information is unclear, write "Not reported".
Do not infer unsupported information.
"""

matrix_response = client.responses.create(
    model=ANSWER_MODEL,
    input=MATRIX_PROMPT,
    tools=[
        {
            "type": "file_search",
            "vector_store_ids": [vector_store.id]
        }
    ]
)

data = json.loads(matrix_response.output_text)
evidence_matrix = pd.DataFrame(data)

display(evidence_matrix)

evidence_matrix.to_csv(
    "Research_Evidence_Matrix.csv",
    index=False,
    encoding="utf-8-sig"
)

print("File created: Research_Evidence_Matrix.csv")